# Chess Assistant — train model FOMO cho ESP32-S3

FOMO (*Faster Objects, More Objects*) là kiểu nhận diện chỉ ra **tâm** vật thể, không ra khung, nhẹ hơn YOLO hàng trăm lần:
MobileNetV2 α=0.35 cắt ở tầng 1/8 + vài lớp 1x1 → lưới xác suất. Bản int8 chỉ khoảng 50–150 KB.

Khác FOMO gốc (lưới 1/8): ở đây có thêm nhánh phóng lớp 1/8 lên rồi ghép với lớp 1/4 → **lưới 40x40** cho ảnh 160px.
Thử với nhãn hoàn hảo trên ChessReD (bàn cờ thật chụp nghiêng), lưới 20x20 chỉ đạt tối đa ~80% ô đúng vì hàng xa camera
mỗi ô cờ chỉ rộng ~1 ô lưới; lưới 40x40 đạt ~99%.

Model học 2 thứ:
- **tâm đế quân** (đúng điểm cần để biết quân đứng ô nào);
- **4 góc bàn cờ** (lớp `corner`) → ESP32 tự tính phép chiếu phối cảnh, không cần OpenCV.

`SCHEME = 'color'` (mặc định): chỉ phân biệt quân Trắng / quân Đen. Đủ để theo dõi ván từ thế khởi đầu, vì `GameTracker`
khớp nước đi theo luật cờ. `'full'`: 12 loại quân (khó hơn nhiều ở ảnh 160px).

**Chuẩn bị (trên máy bạn):**
1. `python -m scripts.prepare_fomo` → tạo `colab_upload/fomo_dataset.zip` (~200 MB)
2. Tải `fomo_dataset.zip` lên Google Drive vào **`MyDrive/ChessAssistant/`**

**Trên Colab:** `Runtime → Change runtime type → T4 GPU`, rồi chạy lần lượt từng ô.

**Kết quả** (trong `MyDrive/ChessAssistant/fomo_<scheme>_<size>/`):
- `fomo_int8.tflite` — model đã lượng tử hoá int8;
- `esp32/` (và `fomo_esp32.zip`) — `fomo_model_data.cc/.h` (model dạng mảng C) + `fomo_settings.h` (kích thước, ngưỡng,
  tham số lượng tử) → chép vào `firmware/esp32s3_fomo/main/model/`.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Kết nối Drive và giải nén dữ liệu vào ổ của Colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, shutil, sys, zipfile
from collections import Counter
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/ChessAssistant')
DATASET_ZIP = DRIVE_DIR / 'fomo_dataset.zip'
DATA_DIR = Path('/content/fomo')
assert DATASET_ZIP.exists(), f'Không thấy {DATASET_ZIP}. Chạy python -m scripts.prepare_fomo rồi tải file lên Drive.'

if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
with zipfile.ZipFile(DATASET_ZIP) as z:
    z.extractall(DATA_DIR)
sys.path.insert(0, str(DATA_DIR))
import fomo  # = chess_assistant/fomo.py trên máy bạn (decode, góc, chiếu ô, tăng cường dữ liệu)

meta = json.loads((DATA_DIR / 'labels.json').read_text())
records = meta['records']
for (source, split), n in sorted(Counter((r['source'], r['split']) for r in records).items()):
    print(f'{source:12s} {split:5s} {n:5d} ảnh')

## 3. Cấu hình
- `IMG_SIZE`: 160 là cân bằng tốt cho ESP32-S3. Chỉ 96/128/160/192/224 có trọng số ImageNet sẵn.
- `STRIDE`: 4 = lưới 1/4 (khuyên dùng, có nhánh nâng phân giải); 8 = FOMO gốc (nhẹ hơn chút, kém với bàn cờ chụp nghiêng).
- `OBJECT_WEIGHT`: ô lưới có quân/góc ít hơn ô nền rất nhiều, nên phạt sai ở đó nặng hơn.
- `SOURCE_REPEAT`: lặp nguồn ít ảnh (ChessReD, ảnh tự chụp) để model không chỉ học cờ 2D.
- `RESUME = True`: train tiếp từ `best.keras` đã lưu trên Drive (khi Colab bị ngắt).

In [ ]:
SCHEME = 'color'      # 'color': Trắng / Đen / góc  —  'full': 12 loại quân + góc
IMG_SIZE = 160
STRIDE = 4
ALPHA = 0.35
EPOCHS = 60
BATCH = 32
LR = 1e-3
OBJECT_WEIGHT = 50.0
SOURCE_REPEAT = {'chessred2k': 3, 'chess2d': 1, 'game3d': 20}
RESUME = False

CLASS_NAMES = fomo.SCHEMES[SCHEME]          # kênh 1.. (kênh 0 = nền)
NUM_CH = len(CLASS_NAMES) + 1
CORNER_CH = len(CLASS_NAMES)
GRID = IMG_SIZE // STRIDE
RUN_DIR = DRIVE_DIR / f'fomo_{SCHEME}_{IMG_SIZE}_s{STRIDE}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Kênh:', ('background',) + CLASS_NAMES, '| lưới', GRID, 'x', GRID, '| lưu vào', RUN_DIR)

## 4. Đọc dữ liệu
Mỗi ảnh train được cắt thành khung vuông quanh bàn cờ (bàn chiếm 50–97% khung, xoay bất kỳ, lật ngang), rồi đổi sáng,
mờ, nhiễu, nén JPEG — giống camera OV2640 trên kính. Ảnh val/test: bàn cờ ở giữa, chiếm 80% khung.

In [ ]:
import cv2
import numpy as np
import keras
import tensorflow as tf

splits = {s: [r for r in records if r['split'] == s] for s in ('train', 'val', 'test')}


def load(rec):
    img = cv2.imread(str(DATA_DIR / rec['file']))
    return img, [tuple(p) for p in rec['points']]


def eval_view(rec):
    img, pts = load(rec)
    return fomo.board_view(img, pts, IMG_SIZE)


class FomoData(keras.utils.PyDataset):
    def __init__(self, recs, train, **kwargs):
        super().__init__(**kwargs)
        self.train = train
        self.recs = [r for r in recs for _ in range(SOURCE_REPEAT.get(r['source'], 1) if train else 1)]
        self.order = np.arange(len(self.recs))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.recs) / BATCH))

    def on_epoch_end(self):
        if self.train:
            np.random.default_rng().shuffle(self.order)

    def __getitem__(self, idx):
        ids = self.order[idx * BATCH:(idx + 1) * BATCH]
        x = np.zeros((len(ids), IMG_SIZE, IMG_SIZE, 3), np.float32)
        y = np.zeros((len(ids), GRID, GRID), np.int32)
        rng = np.random.default_rng()
        for k, i in enumerate(ids):
            if self.train:
                img, pts = fomo.random_view(*load(self.recs[i]), IMG_SIZE, rng)
            else:
                img, pts = eval_view(self.recs[i])
            x[k] = fomo.preprocess(img)
            y[k] = fomo.make_target(pts, GRID, SCHEME)
        return x, y


train_data = FomoData(splits['train'], True, workers=8, use_multiprocessing=False)
val_data = FomoData(splits['val'], False, workers=8, use_multiprocessing=False)
print(len(train_data.recs), 'ảnh train / epoch (đã lặp nguồn) |', len(val_data.recs), 'ảnh val')

Xem thử vài ảnh train sau tăng cường (chấm đỏ = góc, trắng/đen = quân):

In [ ]:
import matplotlib.pyplot as plt

xb, yb = train_data[0]
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, x, y in zip(axes.flat, xb, yb):
    ax.imshow((x + 1) / 2)
    ii, jj = np.nonzero(y)
    for i, j in zip(ii, jj):
        name = CLASS_NAMES[y[i, j] - 1]
        c = 'red' if y[i, j] == CORNER_CH else ('white' if name in ('white',) or name.isupper() else 'black')
        ax.scatter((j + 0.5) * STRIDE, (i + 0.5) * STRIDE, s=25, c=c, edgecolors='yellow')
    ax.axis('off')
plt.tight_layout(); plt.show()

## 5. Model FOMO và train

In [ ]:
from keras import layers


def build_fomo():
    weights = 'imagenet' if IMG_SIZE in (96, 128, 160, 192, 224) else None
    base = keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), alpha=ALPHA,
                                          include_top=False, weights=weights)
    x = base.get_layer('block_6_expand_relu').output        # độ phân giải 1/8
    x = layers.Conv2D(32, 1, activation='relu', name='fomo_head')(x)
    if STRIDE == 4:  # phóng lên 1/4 và ghép với đặc trưng 1/4 (chi tiết vị trí tốt hơn)
        x = layers.UpSampling2D(2, interpolation='nearest', name='fomo_up')(x)
        x = layers.Concatenate(name='fomo_cat')([x, base.get_layer('block_3_expand_relu').output])
        x = layers.Conv2D(32, 1, activation='relu', name='fomo_head4')(x)
    out = layers.Conv2D(NUM_CH, 1, activation='softmax', name='fomo_out')(x)
    return keras.Model(base.input, out, name='fomo')


def weighted_ce(y_true, y_pred):
    y_true = tf.cast(y_true, tf.int32)
    ce = keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    w = tf.where(y_true > 0, OBJECT_WEIGHT, 1.0)
    return tf.reduce_mean(ce * w)


ckpt = RUN_DIR / 'best.keras'
model = keras.models.load_model(ckpt, compile=False) if RESUME and ckpt.exists() else build_fomo()
print(f'{model.count_params():,} tham số | đầu ra {model.output_shape}')
model.compile(optimizer=keras.optimizers.Adam(keras.optimizers.schedules.CosineDecay(LR, len(train_data) * EPOCHS)),
              loss=weighted_ce)
history = model.fit(
    train_data, validation_data=val_data, epochs=EPOCHS,
    callbacks=[keras.callbacks.ModelCheckpoint(str(ckpt), monitor='val_loss', save_best_only=True),
               keras.callbacks.CSVLogger(str(RUN_DIR / 'log.csv'), append=RESUME)])
model = keras.models.load_model(ckpt, compile=False)
print('Model tốt nhất:', ckpt)

## 6. Đánh giá
- **P / R**: điểm dự đoán đúng kênh và cách điểm thật ≤ 1,5 ô lưới.
- **ô (góc thật)**: độ đúng 64 ô khi dùng 4 góc thật — chỉ đo phần nhận quân.
- **ô (đầu-cuối)**: dùng cả 4 góc model tìm được, *đúng như ESP32 sẽ làm* (thử 4 hướng xoay, vì hướng thật xác định
  sau bằng thế khởi đầu). Đây là con số quan trọng nhất.
- **đủ góc**: tỉ lệ ảnh tìm được ≥ 3 góc.

Ngưỡng được chọn trên tập val rồi dùng cho tập test.

In [ ]:
from collections import defaultdict

THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]


def views_of(recs):
    vs = [eval_view(r) for r in recs]
    return np.stack([fomo.preprocess(v) for v, _ in vs]), [p for _, p in vs]


def keras_predict(x):
    return model.predict(x, batch_size=64, verbose=0)


def match(dets, pts, radius=1.5 / GRID):
    tp = 0
    truth = [(fomo.channel_of(c, SCHEME), x, y) for c, x, y in pts]
    used = [False] * len(truth)
    for d in dets:
        best, best_k = radius, None
        for k, (c, x, y) in enumerate(truth):
            dist = np.hypot(d.x - x, d.y - y)
            if not used[k] and c == d.channel and dist <= best:
                best, best_k = dist, k
        if best_k is not None:
            used[best_k] = True
            tp += 1
    return tp, len(dets) - tp, len(truth) - tp


def evaluate(recs, probs, pts_list, threshold):
    s = defaultdict(lambda: defaultdict(float))
    for rec, p, pts in zip(recs, probs, pts_list):
        dets = fomo.decode(p, threshold)
        st = s[rec['source']]
        tp, fp, fn = match(dets, pts)
        st['tp'] += tp; st['fp'] += fp; st['fn'] += fn; st['n'] += 1
        truth = fomo.target_occupancy(pts, SCHEME)
        if truth is None:
            continue
        occ_true, corners_true = truth
        st['sq_gt'] += fomo.square_accuracy(fomo.occupancy(dets, corners_true, SCHEME), occ_true)
        found = fomo.complete_corners([(d.x, d.y) for d in dets if d.channel == CORNER_CH])
        if found is not None:
            st['corners'] += 1
            occ = fomo.occupancy(dets, fomo.order_corners(found), SCHEME)
            st['sq_e2e'] += fomo.square_accuracy(occ, occ_true, any_rotation=True)
        st['m'] += 1
    rows = {}
    for src, st in s.items():
        rows[src] = {'P': st['tp'] / max(st['tp'] + st['fp'], 1), 'R': st['tp'] / max(st['tp'] + st['fn'], 1),
                     'sq_gt': st['sq_gt'] / max(st['m'], 1), 'sq_e2e': st['sq_e2e'] / max(st['m'], 1),
                     'corners': st['corners'] / max(st['m'], 1), 'n': int(st['n'])}
    return rows


def show(rows, title):
    print(title)
    for src, r in rows.items():
        print(f"  {src:12s} {r['n']:5d} ảnh | P {r['P']:.3f} R {r['R']:.3f} | ô (góc thật) {r['sq_gt']:.3f} "
              f"| ô (đầu-cuối) {r['sq_e2e']:.3f} | đủ góc {r['corners']:.3f}")


def mean_e2e(rows):
    return float(np.mean([r['sq_e2e'] for r in rows.values()]))  # trung bình theo nguồn, không để cờ 2D lấn át


x_val, pts_val = views_of(splits['val'])
p_val = keras_predict(x_val)
scores = {t: mean_e2e(evaluate(splits['val'], p_val, pts_val, t)) for t in THRESHOLDS}
THRESHOLD = max(scores, key=scores.get)
print('Ô đúng (đầu-cuối) trên val theo ngưỡng:', {t: round(v, 3) for t, v in scores.items()}, '→ chọn', THRESHOLD)

x_test, pts_test = views_of(splits['test'])
rows_float = evaluate(splits['test'], keras_predict(x_test), pts_test, THRESHOLD)
show(rows_float, f'TEST (float32, ngưỡng {THRESHOLD})')

## 7. Lượng tử hoá int8 (cho ESP32) và kiểm tra lại
ESP32-S3 chạy nhanh nhất với model int8 (TFLite Micro + ESP-NN). Độ đúng sau lượng tử hoá thường chỉ giảm chút ít —
nếu giảm nhiều, tăng số ảnh mẫu `REP_SAMPLES`.

In [ ]:
REP_SAMPLES = 300


def representative():
    rng = np.random.default_rng(0)
    for i in rng.choice(len(splits['train']), min(REP_SAMPLES, len(splits['train'])), replace=False):
        img, pts = load(splits['train'][i])
        view, _ = fomo.board_view(img, pts, IMG_SIZE, fill=float(rng.uniform(0.55, 0.95)))
        yield [fomo.preprocess(view)[None]]


@tf.function(input_signature=[tf.TensorSpec([1, IMG_SIZE, IMG_SIZE, 3], tf.float32)])
def serve(x):
    return model(x, training=False)


converter = tf.lite.TFLiteConverter.from_concrete_functions([serve.get_concrete_function()], model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_int8 = converter.convert()
(RUN_DIR / 'fomo_int8.tflite').write_bytes(tflite_int8)
print(f'fomo_int8.tflite: {len(tflite_int8) / 1024:.0f} KB')

interp = tf.lite.Interpreter(model_content=tflite_int8)
interp.allocate_tensors()
inp, outp = interp.get_input_details()[0], interp.get_output_details()[0]
IN_SCALE, IN_ZP = inp['quantization']
OUT_SCALE, OUT_ZP = outp['quantization']
try:
    OPS = sorted({d['op_name'] for d in interp._get_ops_details()})
except Exception:
    OPS = []
print('Vào', inp['shape'], inp['dtype'].__name__, (IN_SCALE, IN_ZP), '| ra', outp['shape'], (OUT_SCALE, OUT_ZP))
print('Phép toán cần đăng ký trong firmware:', OPS)


def int8_predict(x):
    out = []
    for xi in x:
        q = np.clip(np.round(xi / IN_SCALE + IN_ZP), -128, 127).astype(np.int8)[None]
        interp.set_tensor(inp['index'], q)
        interp.invoke()
        out.append((interp.get_tensor(outp['index'])[0].astype(np.float32) - OUT_ZP) * OUT_SCALE)
    return np.stack(out)


rows_int8 = evaluate(splits['test'], int8_predict(x_test), pts_test, THRESHOLD)
show(rows_int8, f'TEST (int8, ngưỡng {THRESHOLD})')

## 8. Xuất mã C cho ESP32
Tạo `fomo_model_data.cc/.h` (model dạng mảng byte, căn lề 16) và `fomo_settings.h`. Chép cả 3 file vào
`firmware/esp32s3_fomo/main/model/` rồi build firmware (xem `firmware/esp32s3_fomo/README.md`).

In [ ]:
import datetime

OUT = RUN_DIR / 'esp32'
OUT.mkdir(exist_ok=True)

body = ',\n'.join('  ' + ', '.join(f'0x{b:02x}' for b in tflite_int8[i:i + 16]) for i in range(0, len(tflite_int8), 16))
(OUT / 'fomo_model_data.h').write_text(
    '#pragma once\n\n'
    'extern const unsigned char g_fomo_model[];\n'
    'extern const unsigned int g_fomo_model_len;\n')
(OUT / 'fomo_model_data.cc').write_text(
    '// Sinh bởi notebooks/train_fomo_colab.ipynb — không sửa tay.\n'
    '#include "fomo_model_data.h"\n\n'
    f'alignas(16) const unsigned char g_fomo_model[] = {{\n{body}\n}};\n'
    f'const unsigned int g_fomo_model_len = {len(tflite_int8)};\n')

symbols = '.WBX' if SCHEME == 'color' else '.' + fomo.PIECE_CLASSES + 'X'
labels = ', '.join(f'"{n}"' for n in ('background',) + CLASS_NAMES)
test_e2e = {k: round(v['sq_e2e'], 3) for k, v in rows_int8.items()}
(OUT / 'fomo_settings.h').write_text(f'''#pragma once
// Sinh bởi notebooks/train_fomo_colab.ipynb lúc {datetime.datetime.now():%Y-%m-%d %H:%M} — không sửa tay.
// Scheme: {SCHEME}, lưới 1/{STRIDE} | ô đúng (đầu-cuối, int8, test): {test_e2e}
// Phép toán trong model: {', '.join(OPS)}

#define FOMO_INPUT_SIZE {IMG_SIZE}
#define FOMO_GRID {GRID}
#define FOMO_NUM_CHANNELS {NUM_CH}      // gồm kênh 0 = nền
#define FOMO_CORNER_CHANNEL {CORNER_CH}
#define FOMO_THRESHOLD {THRESHOLD}f
#define FOMO_INPUT_SCALE {IN_SCALE:.10g}f
#define FOMO_INPUT_ZERO_POINT {IN_ZP}
#define FOMO_OUTPUT_SCALE {OUT_SCALE:.10g}f
#define FOMO_OUTPUT_ZERO_POINT {OUT_ZP}

// Ký hiệu in ra Serial cho từng kênh: '.' = trống, X = góc bàn cờ.
#define FOMO_SYMBOLS "{symbols}"
static const char *const FOMO_LABELS[FOMO_NUM_CHANNELS] = {{{labels}}};
''')
(RUN_DIR / 'metrics.json').write_text(json.dumps(
    {'scheme': SCHEME, 'img_size': IMG_SIZE, 'stride': STRIDE, 'threshold': THRESHOLD, 'float': rows_float, 'int8': rows_int8,
     'tflite_kb': len(tflite_int8) / 1024, 'ops': OPS}, indent=2, ensure_ascii=False))
shutil.make_archive(str(DRIVE_DIR / 'fomo_esp32'), 'zip', OUT)
print('Đã lưu', OUT, 'và', DRIVE_DIR / 'fomo_esp32.zip')
for f in sorted(OUT.iterdir()):
    print(f'  {f.name:22s} {f.stat().st_size / 1024:7.1f} KB')

## 9. Xem thử dự đoán trên ảnh test (model int8)

In [ ]:
rng = np.random.default_rng(1)
idx = rng.choice(len(x_test), min(8, len(x_test)), replace=False)
probs = int8_predict(x_test[idx])
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, i, p in zip(axes.flat, idx, probs):
    ax.imshow((x_test[i] + 1) / 2)
    dets = fomo.decode(p, THRESHOLD)
    for d in dets:
        name = CLASS_NAMES[d.channel - 1]
        c = 'red' if d.channel == CORNER_CH else ('white' if name == 'white' or name.isupper() else 'black')
        ax.scatter(d.x * IMG_SIZE, d.y * IMG_SIZE, s=40, c=c, edgecolors='yellow')
    found = fomo.complete_corners([(d.x, d.y) for d in dets if d.channel == CORNER_CH])
    if found is not None:
        q = fomo.order_corners(found) * IMG_SIZE
        ax.plot(*np.vstack([q, q[:1]]).T, c='lime', lw=1)
    ax.set_title(f"{splits['test'][i]['source']} | {len(dets)} điểm", fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()